# Optional: forecasting with machine learning
Build a one-hour-ahead temperature predictor using the supplied lag features. At the start of hour $t$, assume observations through $t-1$ are available.

1. Explain why the rolling mean is shifted.
2. Keep the last 20% for testing. Tune a small RandomForestRegressor with `TimeSeriesSplit` on earlier data and compare with persistence.
3. Report MAE and RMSE only for observed targets. Inspect residuals and their ACF. For the ACF, use a contiguous observed segment so missing hours are not silently treated as adjacent observations.
4. Explain why shuffled K-fold validation is unsuitable and why **rolling one-step** evaluation differs from forecasting the whole test interval at once.
5. Predict what must change for a 6- or 24-hour forecast. Features cannot include observations unavailable at that earlier origin.

Ordinary random-forest regression defines no likelihood here, so do not compare it using ARIMA AIC/BIC.

## Create causal predictors
Use the same weather series as the lecture. Forward filling predictors is a simple exercise policy; missing targets stay excluded.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

weather = pd.read_csv('../../Excercise_data/L08/temperature.csv')
weather['Measurement Timestamp'] = pd.to_datetime(
    weather['Measurement Timestamp'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce'
)
weather = weather[weather['Station Name'] == '63rd Street Weather Station']
weather = weather.dropna(subset=['Measurement Timestamp'])
weather = weather.sort_values('Measurement Timestamp').drop_duplicates('Measurement Timestamp', keep='last')
target = weather.set_index('Measurement Timestamp')['Air Temperature'].asfreq('h')
target = pd.to_numeric(target, errors='coerce')
past = target.ffill()
features = pd.DataFrame(index=target.index)
features['lag_1'] = past.shift(1)
features['lag_24'] = past.shift(24)
features['rolling_24'] = past.shift(1).rolling(24).mean()
features['hour_sin'] = np.sin(2 * np.pi * features.index.hour / 24)
features['hour_cos'] = np.cos(2 * np.pi * features.index.hour / 24)
cutoff = target.index[int(0.8 * len(target))]
usable = features.notna().all(axis=1) & target.notna()
X = features.loc[usable]
y = target.loc[usable]
X_train, X_test = X.loc[X.index < cutoff], X.loc[X.index >= cutoff]
y_train, y_test = y.loc[y.index < cutoff], y.loc[y.index >= cutoff]
print(X_train.shape, X_test.shape)

(6307, 5) (1716, 5)


In [2]:
# ADD YOUR CODE HERE
# Use TimeSeriesSplit(n_splits=3), a small grid and random_state=42.
# Persistence predictions are X_test['lag_1'].

### Optional extension
Fit a shallow decision tree using elapsed time only. Can piecewise-constant predictions extrapolate a continuing trend beyond the training period? Predict before plotting.